# 1 First Neural Network

## Phase1: Data Loading and Alignment

- pandas for efficient data manipulation and loading the processed Parquet/CSV files
- numpy for high-performance numerical operations
- gc (Garbage Collector) to manually release memory and prevent RAM overflow during the heavy merging steps

In [ ]:
"""
Created on Tuesday Nov 18 2025

@author: Laura Rueda
"""
import pandas as pd
import numpy as np
import gc
import os

In [7]:
# 1. SETUP
# We define the paths to the files we created in the previous notebook
CLEAN_CSV_DIR = 'data/clean_csv/'
ONE_HOT_DIR = 'data/onehotencoding/'
EMBEDDINGS_DIR = 'data/embeddings/'

os.makedirs(CLEAN_CSV_DIR, exist_ok=True)
os.makedirs(ONE_HOT_DIR, exist_ok=True)
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)


print("--- STEP 1: LOADING DATA ---")

# A. Load the Base File (Patient IDs, Dates, Target)
# We only need these columns to organize the data
print("Loading Base Data (Patients & Time)...")
df_base = pd.read_csv(f'{CLEAN_CSV_DIR}deid_rx_order_final_sql.csv', 
                      usecols=['PATIENT_ID', 'Shifted_date', 'AIM_GROUP'])

# B. Load the Features (The Parquet files)
print("Loading Features (Parquet files)...")
df_meds = pd.read_parquet(f'{EMBEDDINGS_DIR}rx_code_ids.parquet')   # Embeddings Input
df_dose = pd.read_parquet(f'{ONE_HOT_DIR}dose_one_hot.parquet')  # One-Hot Input
df_route = pd.read_parquet(f'{ONE_HOT_DIR}route_one_hot.parquet') # One-Hot Input
df_freq = pd.read_parquet(f'{ONE_HOT_DIR}freq_one_hot.parquet')   # One-Hot Input

print("Data Loaded Successfully.")

# C. Verify Alignment (CRITICAL)
# All files must have exactly the same number of rows
assert len(df_base) == len(df_meds) == len(df_dose) == len(df_route) == len(df_freq)
print(f"Alignment Check Passed: All files have {len(df_base)} rows.")

--- STEP 1: LOADING DATA ---
Loading Base Data (Patients & Time)...
Loading Features (Parquet files)...
Data Loaded Successfully.
Alignment Check Passed: All files have 20436701 rows.


## Phase 2: Preparing the Target (Y)

In [8]:
print("\n--- STEP 2: ENCODING TARGET (Y) ---")

# Check unique values
print("Original Targets:", df_base['AIM_GROUP'].unique())

# Convert to Binary (0 and 1)
# 1_NoD -> 0 (Negative/Healthy)
# 2_T2D -> 1 (Positive/Diabetes)
df_base['TARGET'] = df_base['AIM_GROUP'].map({'1_NoD': 0, '2_T2D': 1})

# Check the balance
print("Target Distribution:")
print(df_base['TARGET'].value_counts())

# Drop the text column to save memory
df_base.drop(columns=['AIM_GROUP'], inplace=True)
print("Target encoded to 0/1.")


--- STEP 2: ENCODING TARGET (Y) ---
Original Targets: ['1_NoD' '2_Type2']
Target Distribution:
TARGET
0.0    16292902
Name: count, dtype: int64
Target encoded to 0/1.


## Phase 3: Merging for Sequence Creation

In [9]:
print("\n--- STEP 3: MERGING FEATURES ---")

# We concatenate (join horizontally) all our dataframes
# axis=1 means "side by side"
full_data = pd.concat([df_base, df_meds, df_dose, df_route, df_freq], axis=1)

# Inspect the result
print("Full Dataset Shape:", full_data.shape)
print(full_data.head())

# Clean up memory (Delete individual parts now that they are in 'full_data')
del df_base, df_meds, df_dose, df_route, df_freq
gc.collect()
print("✅ Data Merged. Memory cleaned.")


--- STEP 3: MERGING FEATURES ---
Full Dataset Shape: (20436701, 97)
   PATIENT_ID         Shifted_date  TARGET  RX_CODE_ID  DOSE_0.025 mg  \
0           1  2015-10-18 15:27:00     0.0        3908              0   
1           1  2015-10-31 19:05:00     0.0        3908              0   
2           1  2016-01-10 17:03:00     0.0        4253              0   
3           1  2016-01-10 17:03:00     0.0        9928              0   
4           1  2016-01-10 17:03:00     0.0        9928              0   

   DOSE_0.05 mg  DOSE_0.1 mg  DOSE_0.2 mg  DOSE_0.4 mg  DOSE_0.5 mg  ...  \
0             0            0            0            0            0  ...   
1             0            0            0            0            0  ...   
2             0            0            0            0            0  ...   
3             0            0            0            0            0  ...   
4             1            0            0            0            0  ...   

   FREQ_OTHER  FREQ_Q12H  FREQ_Q4H 